# **Content planner flow**

Before you get started:

Create a .env file in the directory and store these two values:

- FIRECRAWL_API_KEY="fc-..." (get the firecrawl API key here: https://www.firecrawl.dev/i/api)
- OPENAI_API_KEY="sk-..."

In [24]:
# Importing necessary libraries
import os
import uuid
import yaml
import json
from pathlib import Path
from pydantic import BaseModel
from typing import Optional

# Firecrawl SDK
from firecrawl import FirecrawlApp 

# Importing Crew related components
from crewai import Agent, Task, Crew, LLM

# Importing CrewAI Flow related components
from crewai.flow.flow import Flow, listen, start, router, or_

from dotenv import load_dotenv
load_dotenv()

import nest_asyncio
nest_asyncio.apply()

In [29]:
llm = LLM(
     model="ollama/llama3.2:1b",
     base_url="http://localhost:11434"
 )

#llm = LLM(
#    model="gpt-4o",
#)

In [3]:
blog_post_url = "https://blog.dailydoseofds.com/p/5-chunking-strategies-for-rag"

## Twitter and LinkedIn Planning Crew

In [30]:
# define structured output for twitter and linkedin

class Tweet(BaseModel):
    """Represents an individual tweet in a thread"""
    content: str
    is_hook: bool = False  # Identifies if this is the opening/hook tweet
    media_urls: Optional[list[str]] = []  # Optional media attachments (images, code snippets)

class Thread(BaseModel):
    """Represents a Twitter thread"""
    topic: str  # Main topic/subject of the thread
    tweets: list[Tweet]  # List of tweets in the thread

class LinkedInPost(BaseModel):
    """Represents a LinkedIn post"""
    content: str
    media_url: str # Main image url for the post

In [31]:
# load agent and task configurations from yaml files and tools

from crewai_tools import (
    DirectoryReadTool,
    FileReadTool,
)

# Load agent and task configurations from YAML files
with open('config/planner_agents.yaml', 'r') as f:
    agents_config = yaml.safe_load(f)

with open('config/planner_tasks.yaml', 'r') as f:
    tasks_config = yaml.safe_load(f)

In [32]:
# create agents, their tasks and crew for twitter

draft_analyzer = Agent(config=agents_config['draft_analyzer'], tools=[
    DirectoryReadTool(),
    FileReadTool()
], llm=llm)

twitter_thread_planner = Agent(config=agents_config['twitter_thread_planner'], tools=[
    DirectoryReadTool(),
    FileReadTool()
], llm=llm)

analyze_draft = Task(
  config=tasks_config['analyze_draft'],
  agent=draft_analyzer
)

create_twitter_thread_plan = Task(
  config=tasks_config['create_twitter_thread_plan'],
  agent=twitter_thread_planner,
  output_pydantic=Thread
)

twitter_planning_crew = Crew(
    agents=[draft_analyzer, twitter_thread_planner],
    tasks=[analyze_draft, create_twitter_thread_plan],
    verbose=False
)

In [33]:
# create agents, their tasks and crew for linkedin

linkedin_post_planner = Agent(config=agents_config['linkedin_post_planner'], tools=[
    DirectoryReadTool(),
    FileReadTool()
    ], llm=llm)

create_linkedin_post_plan = Task(
  config=tasks_config['create_linkedin_post_plan'],
  agent=linkedin_post_planner,
  output_pydantic=LinkedInPost
)

linkedin_planning_crew = Crew(
    agents=[draft_analyzer, linkedin_post_planner],
    tasks=[analyze_draft, create_linkedin_post_plan],
    verbose=False
)

In [34]:
# define state for the content planning flow

from firecrawl import Firecrawl


class ContentPlanningState(BaseModel):
  """
  State for the content planning flow
  """
  blog_post_url: str = blog_post_url
  draft_path: Path = "assets/ "
  post_type: str = "twitter"
  path_to_example_threads: str = "assets/example_threads.txt"
  path_to_example_linkedin: str = "assets/example_linkedin.txt"

class CreateContentPlanningFlow(Flow[ContentPlanningState]):
  # No need for AI Agents on this step, so we just use regular Python code
  @start()
  def scrape_blog_post(self):
      print(f"# fetching draft from: {self.state.blog_post_url}")

      app = Firecrawl(
          api_key=os.getenv("FIRECRAWL_API_KEY")
      )

      scrape_result = app.scrape(
          self.state.blog_post_url,
          formats=["markdown", "html"]
      )

      try:
          title = scrape_result.metadata.title
      except:
          title = str(uuid.uuid4())

      self.state.draft_path = f"assets/{title}.md"

      with open(self.state.draft_path, "w", encoding="utf-8") as f:
          f.write(scrape_result.markdown)

      return self.state

  @router(scrape_blog_post)
  def select_platform(self):
    if self.state.post_type == "twitter":
      return "twitter"
    elif self.state.post_type == "linkedin":
      return "linkedin"

  @listen("twitter")
  def twitter_draft(self):
    print(f"# Planning content for: {self.state.draft_path}")
    
    result = twitter_planning_crew.kickoff(inputs={'draft_path': self.state.draft_path, 'path_to_example_threads': self.state.path_to_example_threads})
    
    print(f"# Planned content for {self.state.draft_path}:")
    
    for i, tweet in enumerate(result.pydantic.tweets):
        
        print(f"Tweet {i+1}:")
        print(f"{tweet.content}")
        print(f"Media URLs: {tweet.media_urls}")

        print("-"*100)
    return result
  
  @listen("linkedin")
  def linkedin_draft(self):
    print(f"# Planning content for: {self.state.draft_path}")
    result = linkedin_planning_crew.kickoff(inputs={'draft_path': self.state.draft_path, 'path_to_example_linkedin': self.state.path_to_example_linkedin})
    print(f"# Planned content for {self.state.draft_path}:")
    print(f"{result.pydantic.content}")
    return result

  @listen(or_(twitter_draft, linkedin_draft))
  def save_plan(self, plan):
    with open(f'output/{self.state.draft_path.split("/")[-1]}_{self.state.post_type}.json', 'w') as f:
        json.dump(plan.pydantic.model_dump(), f, indent=2)

In [35]:
# Plot the flow
flow = CreateContentPlanningFlow()
flow.state.post_type = "twitter"

In [21]:
flow.plot()

Router events for 'select_platform' are dynamic or not statically inferable; static visualization may omit event edges.
Static visualization could not match listener triggers {'twitter', 'linkedin'} to explicit router events. Dynamic router values may still trigger these listeners at runtime.


'C:\\Users\\hp\\AppData\\Local\\Temp\\crewai_flow_qkxmiopl\\crewai_flow.html'

In [36]:
flow.state

StateWithId(blog_post_url='https://blog.dailydoseofds.com/p/5-chunking-strategies-for-rag', draft_path='assets/ ', post_type='twitter', path_to_example_threads='assets/example_threads.txt', path_to_example_linkedin='assets/example_linkedin.txt', id='3cb86754-c102-4c97-a848-ccc717616ba9')

In [37]:
flow.kickoff()

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: CreateContentPlanningFlow                                                                                │
│  ID: 3cb86754-c102-4c97-a848-ccc717616ba9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: CreateContentPlanningFlow                                                                                │
│  ID: 3cb86754-c102-4c97-a848-ccc717616ba9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# fetching draft from: https://blog.dailydoseofds.com/p/5-chunking-strategies-for-rag


╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: scrape_blog_post                                                                                       │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: scrape_blog_post                                                                                       │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

# Planned content for assets/5 Chunking Strategies For RAG - by Avi Chawla.md:
Tweet 1:
Start breaking down RAG programs into smaller pieces by identifying core ideas.
Media URLs: []
----------------------------------------------------------------------------------------------------
Tweet 2:
Apply chunking techniques to improve development efficiency!
Media URLs: []
----------------------------------------------------------------------------------------------------
Tweet 3:
Real-world examples show how best practices can boost productivity. Read more: https://www.rag.org/chunking-strategies-step-by-step [link]
Media URLs: ['https://www.github.io/avichawl/chunking-st strategies']
----------------------------------------------------------------------------------------------------
Tweet 4:
Best practice for writing code snippets like this one: https://www.example.com/how-to-implement-chunking-strategies [link]
Media URLs: []
----------------------------------------------------------------

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: twitter_draft                                                                                          │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: save_plan                                                                                              │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: save_plan                                                                                              │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: CreateContentPlanningFlow                                                                                │
│  ID: 3cb86754-c102-4c97-a848-ccc717616ba9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
flow.state.post_type = "linkedin"
flow.kickoff()

 Flow started with ID: 5973fef0-9201-489f-818c-a40959f64f6e
# fetching draft from: https://blog.dailydoseofds.com/p/5-chunking-strategies-for-rag
# Planning content for: assets/5 Chunking Strategies For RAG - by Avi Chawla.md
# Planned content for assets/5 Chunking Strategies For RAG - by Avi Chawla.md:
🔍 **5 Chunking Strategies For RAG** 🚀

Understanding how to effectively handle large-scale data is pivotal for any data-driven business. In this exploration, we delve into '5 Chunking Strategies For RAG' to unravel the complexities and uncover strategies that illuminate the path to efficient data processing.

Effective data chunking is not just a technical hurdle; it's a business imperative. Without it, systems can become inefficient, data retrieval times can skyrocket, and ultimately, the user experience can falter. These challenges not only hinder operational efficiency but can significantly impact the competitive edge of your business.

So, what are the solutions and technical approa